# Аудит аплифта дорожных карт DD

## tl;dr

- Проверено 204 логических roadmap items; 181 имеют числовой uplift.
- Выявлено 19 групп `юнит × продукт × канонический набор блоков`, где округлённый Excel total превышает округлённый теоретический максимум; на листе детализации 27 исходных items.
- По флагам сумма Excel total — 174,9 п.п., сумма теоретических максимумов — 109,1 п.п., арифметическая сумма завышения — 65,8 п.п. Эти суммы не являются портфельной оценкой при пересекающихся наборах блоков.
- На лист «Не сопоставлено» вынесены 3 items с числовым uplift и пустым блоком: Детская карта, Трансграничные переводы и Трансграничный эквайринг. Ещё 15 несопоставленных items без uplift учтены только как фоновая проверка в методике.
- Обязательная проверка `ДСЖ ПК / Знание ключевых метрик` пройдена: Excel 30,0 п.п. против настоящего максимума 5,1 п.п.

> Важно: рассчитанное значение — теоретический максимум для всего блока, а не прогноз конкретного мероприятия.

## Context & Methods

### Key Assumptions

- Единица дорожной карты — логический item по merged-cell semantics текущего приложения: физические строки внутри объединённого диапазона G (или D/F/H для item без uplift) объединяются в одно `planned_activity`.
- Авторитетное контрольное зерно книги — 204 roadmap items, из них 181 с числовым uplift; ноутбук проверяет эти количества assertion-ами.
- Обычные пустые блоки не заполняются и не угадываются.
- Авторитетный уровень расчёта DD — `продукт × блок` из JSON metrics.
- Все вычисления выполняются через `Decimal`; сравнение делается после `ROUND_HALF_UP` до 1 знака.
- Кэшированное числовое значение Excel-формулы используется как `expected_uplift`; нечисловые и пустые значения дают нулевой вклад в сумму группы.

### 1. Setup and visible parameters

Для повторного запуска нужны Python 3.11 и `openpyxl==3.1.5`. Ноутбук создан и исполнен через `nbformat`/`nbclient`; исходные XLSX и JSON читаются без изменений.

In [1]:
from __future__ import annotations

from collections import Counter, defaultdict
from datetime import date
from decimal import Decimal, ROUND_HALF_UP
from pathlib import Path
import json
import math
import re
import unicodedata

from openpyxl import Workbook, load_workbook
from openpyxl.formatting.rule import CellIsRule
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import get_column_letter, range_boundaries

AUDIT_AS_OF = date(2026, 8, 24)
SOURCE_XLSX_REL = Path("Дорожные карты по повышению рейтинга DD.xlsx")
SOURCE_JSON_REL = Path("gravity-app/public/report-data.json")
OUTPUT_XLSX_REL = Path("artifacts/Аудит_аплифта_дорожных_карт_DD.xlsx")

repo_candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
REPO_ROOT = next(
    (path.resolve() for path in repo_candidates
     if (path / SOURCE_XLSX_REL).exists() and (path / SOURCE_JSON_REL).exists()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Не удалось найти корень репозитория с обоими источниками")

SOURCE_XLSX = REPO_ROOT / SOURCE_XLSX_REL
SOURCE_JSON = REPO_ROOT / SOURCE_JSON_REL
OUTPUT_XLSX = REPO_ROOT / OUTPUT_XLSX_REL
OUTPUT_XLSX.parent.mkdir(parents=True, exist_ok=True)

ONE_DECIMAL = Decimal("0.1")
HUNDRED = Decimal("100")

print(f"Repository root: {REPO_ROOT}")
print(f"Roadmap source: {SOURCE_XLSX_REL}")
print(f"DD source: {SOURCE_JSON_REL}")
print(f"Output: {OUTPUT_XLSX_REL}")

Repository root: /Users/roman/Documents/Codex/DD-dev
Roadmap source: Дорожные карты по повышению рейтинга DD.xlsx
DD source: gravity-app/public/report-data.json
Output: artifacts/Аудит_аплифта_дорожных_карт_DD.xlsx


### 2. Deterministic normalization and mappings

Псевдонимы блоков заданы требованиями. Псевдонимы юнита/продукта ограничены прозрачными различиями между подписями дорожной карты и справочником JSON; они перечисляются в итоговой методике.

In [2]:
def normalize_text(value) -> str:
    if value is None:
        return ""
    text = unicodedata.normalize("NFKC", str(value)).replace("\xa0", " ")
    return re.sub(r"\s+", " ", text).strip()


def as_decimal(value, *, allow_none: bool = False):
    if value is None or value == "":
        if allow_none:
            return None
        raise ValueError("Пустое число")
    if isinstance(value, bool):
        raise TypeError("Boolean не является числовым аплифтом")
    if isinstance(value, Decimal):
        return value
    if isinstance(value, (int, float)):
        if isinstance(value, float) and not math.isfinite(value):
            raise ValueError(f"Неконечное число: {value}")
        return Decimal(str(value))
    return Decimal(normalize_text(value).replace(",", "."))


def round_1(value: Decimal) -> Decimal:
    return value.quantize(ONE_DECIMAL, rounding=ROUND_HALF_UP)


UNIT_ALIASES = {
    "СВР": "CBP",
}

PRODUCT_ALIASES_RAW = {
    ("CBP", "Вклады + НС"): "Вклады+НС",
    ("CBP", "ПК"): "Потребительский кредит",
    ("CX", "ПУ СберПремьер"): "Пакет услуг СберПремьер",
    ("CX", "ПУ СберПервый"): "Пакет услуг СберПервый",
    ("CX", "TA"): "Top Affluent",
    ("PC", "Выписки и справки"): "Выписки, справки",
    ("ДомКлик", "Сделка вторичка, Загородка"): "Сделка вторичка",
    ("ДомКлик", "Сделка ИЖС, Загородка"): "Сделка ИЖС",
}
PRODUCT_ALIASES = {
    (normalize_text(unit), normalize_text(source)): normalize_text(target)
    for (unit, source), target in PRODUCT_ALIASES_RAW.items()
}

BLOCK_ORDER = [
    "Знание ключевых метрик", "Цели", "Воронка привлечения", "Воронка оттока",
    "Алерты", "Механики", "Гипотезы и инициативы", "Клиентский опыт",
    "Воронка использования", "Воронка онбординга", "Воронка продаж",
    "Воронка входа в канал", "Данные",
]
BLOCK_ORDER_INDEX = {name: index for index, name in enumerate(BLOCK_ORDER)}

BLOCK_ALIASES_RAW = {
    "CX Score": ("Клиентский опыт",),
    "Цели уровня ЛЮ/ЛТ": ("Цели",),
    "Гипотезы": ("Гипотезы и инициативы",),
    "Знание ключевых метрик\nЦели": ("Знание ключевых метрик", "Цели"),
}
BLOCK_ALIASES = {
    normalize_text(source): tuple(targets)
    for source, targets in BLOCK_ALIASES_RAW.items()
}
KNOWN_BLOCK_BY_NORMALIZED = {normalize_text(name): name for name in BLOCK_ORDER}


def canonical_block_set(raw_block):
    normalized = normalize_text(raw_block)
    if not normalized:
        return None, "Пустой блок"
    if normalized in BLOCK_ALIASES:
        blocks = BLOCK_ALIASES[normalized]
    elif normalized in KNOWN_BLOCK_BY_NORMALIZED:
        blocks = (KNOWN_BLOCK_BY_NORMALIZED[normalized],)
    else:
        return None, "Блок не сопоставлен"
    return tuple(sorted(set(blocks), key=lambda name: BLOCK_ORDER_INDEX[name])), None

print("Unit aliases:", UNIT_ALIASES)
print("Product aliases:", PRODUCT_ALIASES_RAW)
print("Block aliases:", BLOCK_ALIASES_RAW)

Unit aliases: {'СВР': 'CBP'}
Product aliases: {('CBP', 'Вклады + НС'): 'Вклады+НС', ('CBP', 'ПК'): 'Потребительский кредит', ('CX', 'ПУ СберПремьер'): 'Пакет услуг СберПремьер', ('CX', 'ПУ СберПервый'): 'Пакет услуг СберПервый', ('CX', 'TA'): 'Top Affluent', ('PC', 'Выписки и справки'): 'Выписки, справки', ('ДомКлик', 'Сделка вторичка, Загородка'): 'Сделка вторичка', ('ДомКлик', 'Сделка ИЖС, Загородка'): 'Сделка ИЖС'}
Block aliases: {'CX Score': ('Клиентский опыт',), 'Цели уровня ЛЮ/ЛТ': ('Цели',), 'Гипотезы': ('Гипотезы и инициативы',), 'Знание ключевых метрик\nЦели': ('Знание ключевых метрик', 'Цели')}


## Data

### 3. Load and validate authoritative DD metrics

In [3]:
with SOURCE_JSON.open("r", encoding="utf-8") as source_file:
    report_data = json.load(source_file, parse_float=Decimal, parse_int=Decimal)

products = report_data.get("products", [])
product_index = {}
duplicate_product_keys = []
included_metric_records = []
null_included_values = []
value_above_max = []
negative_values = []

for product in products:
    key = (normalize_text(product.get("unit")), normalize_text(product.get("name")))
    if key in product_index:
        duplicate_product_keys.append(key)
    product_index[key] = product
    for block in product.get("metrics", []):
        for metric in block.get("metrics", []):
            max_value = as_decimal(metric.get("max_value"), allow_none=True)
            included = (
                max_value is not None
                and max_value > 0
                and metric.get("is_applicabble_flg") is not False
                and metric.get("excluded_from_index") is not True
                and metric.get("dd_calculation_flg") != 0
            )
            if not included:
                continue
            value = as_decimal(metric.get("value"), allow_none=True)
            record = {
                "unit": key[0], "product": key[1], "block": normalize_text(block.get("name")),
                "metric": normalize_text(metric.get("name")), "value": value, "max_value": max_value,
            }
            included_metric_records.append(record)
            if value is None:
                null_included_values.append(record)
            else:
                if value > max_value:
                    value_above_max.append(record)
                if value < 0:
                    negative_values.append(record)

assert not duplicate_product_keys, f"Дубли product×unit в JSON: {duplicate_product_keys}"
assert not null_included_values, f"У включённых метрик есть пустые value: {null_included_values[:3]}"

print({
    "json_products": len(products),
    "included_metrics": len(included_metric_records),
    "duplicate_product_keys": len(duplicate_product_keys),
    "null_included_values": len(null_included_values),
    "value_above_max": len(value_above_max),
    "negative_values": len(negative_values),
})
if value_above_max:
    print("Примеры value > max_value:", value_above_max[:5])

{'json_products': 80, 'included_metrics': 2550, 'duplicate_product_keys': 0, 'null_included_values': 0, 'value_above_max': 0, 'negative_values': 0}


### 4. Read logical roadmap items with the application’s merged-cell semantics

Алгоритм полностью повторяет `build_calc_report.read_roadmap_workbook()`: распознаёт лист по заголовкам, создаёт item только на стартовой строке логического merge-диапазона и объединяет непустые физические строки E в одно `planned_activity`. Обычные пустые блоки остаются пустыми.

In [4]:
roadmap_formula_wb = load_workbook(SOURCE_XLSX, data_only=False, read_only=False)
roadmap_values_wb = load_workbook(SOURCE_XLSX, data_only=True, read_only=False)


def clean_roadmap_text(value) -> str:
    return "" if value is None else str(value).strip()


def normalize_lookup_key(value) -> str:
    return re.sub(r"\s+", " ", clean_roadmap_text(value)).casefold()


def normalize_roadmap_key(value) -> str:
    normalized = unicodedata.normalize("NFKC", clean_roadmap_text(value)).casefold().replace("ё", "е")
    return re.sub(r"[^0-9a-zа-я]+", "", normalized)


def roadmap_unit_name(sheet_name) -> str:
    source = clean_roadmap_text(sheet_name)
    for alias, canonical in UNIT_ALIASES.items():
        if normalize_roadmap_key(source) == normalize_roadmap_key(alias):
            return canonical
    return source


def roadmap_profile_name(unit: str, source_name) -> str:
    name = re.sub(r"\s+", " ", clean_roadmap_text(source_name))
    for (alias_unit, alias_name), canonical in PRODUCT_ALIASES_RAW.items():
        if (
            normalize_roadmap_key(unit) == normalize_roadmap_key(alias_unit)
            and normalize_roadmap_key(name) == normalize_roadmap_key(alias_name)
        ):
            return canonical
    return name


def roadmap_cell_value(worksheet, row: int, column: int):
    for merged_range in worksheet.merged_cells.ranges:
        if (
            merged_range.min_row <= row <= merged_range.max_row
            and merged_range.min_col <= column <= merged_range.max_col
        ):
            return worksheet.cell(merged_range.min_row, merged_range.min_col).value
    return worksheet.cell(row, column).value


def roadmap_merged_end_row(worksheet, row: int, column: int) -> int:
    for merged_range in worksheet.merged_cells.ranges:
        if (
            merged_range.min_row <= row <= merged_range.max_row
            and merged_range.min_col <= column <= merged_range.max_col
        ):
            return merged_range.max_row
    return row


def roadmap_merged_start_row(worksheet, row: int, column: int) -> int:
    for merged_range in worksheet.merged_cells.ranges:
        if (
            merged_range.min_row <= row <= merged_range.max_row
            and merged_range.min_col <= column <= merged_range.max_col
        ):
            return merged_range.min_row
    return row


def is_roadmap_sheet(worksheet) -> bool:
    return (
        normalize_lookup_key(worksheet.cell(1, 2).value).startswith("продукт")
        and normalize_lookup_key(worksheet.cell(1, 3).value) == "квартал"
        and normalize_lookup_key(worksheet.cell(1, 4).value) == "блок для развития"
        and normalize_lookup_key(worksheet.cell(1, 5).value) == "планируемое мероприятие"
        and normalize_lookup_key(worksheet.cell(1, 7).value).startswith("ожидание прироста индекса dd, п.п")
    )


roadmap_items = []
recognized_sheets = 0
source_numeric_uplifts = 0
source_formula_uplifts = 0

for ws_values in roadmap_values_wb.worksheets:
    if not is_roadmap_sheet(ws_values):
        continue
    recognized_sheets += 1
    ws_formula = roadmap_formula_wb[ws_values.title]
    unit = roadmap_unit_name(ws_values.title)

    for source_row in range(2, ws_values.max_row + 1):
        raw_uplift = ws_values.cell(source_row, 7).value
        has_uplift = raw_uplift is not None and clean_roadmap_text(raw_uplift) != ""
        uplift_decimal = None
        if has_uplift:
            try:
                uplift_decimal = as_decimal(raw_uplift)
            except (TypeError, ValueError, ArithmeticError) as error:
                raise ValueError(
                    f"Некорректный ожидаемый прирост в {ws_values.title}!G{source_row}: {raw_uplift!r}"
                ) from error
            source_numeric_uplifts += 1
            formula_value = ws_formula.cell(source_row, 7).value
            if isinstance(formula_value, str) and formula_value.startswith("="):
                source_formula_uplifts += 1
        else:
            raw_activity = ws_values.cell(source_row, 5).value
            if not isinstance(raw_activity, str) or not clean_roadmap_text(raw_activity):
                continue
            logical_columns = (4, 6, 8)
            if any(
                roadmap_merged_start_row(ws_values, source_row, column) != source_row
                for column in logical_columns
            ):
                continue

        source_name = clean_roadmap_text(roadmap_cell_value(ws_values, source_row, 2))
        if not source_name:
            raise ValueError(
                f"Не указана команда/продукт для ожидаемого прироста в {ws_values.title}!G{source_row}"
            )
        if normalize_lookup_key(source_name).startswith("продукт"):
            continue

        merged_end_row = (
            roadmap_merged_end_row(ws_values, source_row, 7)
            if has_uplift
            else max(
                roadmap_merged_end_row(ws_values, source_row, column)
                for column in (4, 6, 8)
            )
        )
        activity_parts = [
            activity
            for activity_row in range(source_row, merged_end_row + 1)
            if (activity := clean_roadmap_text(ws_values.cell(activity_row, 5).value))
        ]
        profile_name = roadmap_profile_name(unit, source_name)
        product_json_name = PRODUCT_ALIASES.get(
            (normalize_text(unit), normalize_text(source_name)),
            normalize_text(profile_name),
        )
        roadmap_items.append({
            "unit_source": unit,
            "unit_json": normalize_text(unit),
            "product_source": source_name,
            "product_json_name": product_json_name,
            "block_source": clean_roadmap_text(roadmap_cell_value(ws_values, source_row, 4)),
            "activity": "\n".join(activity_parts),
            "uplift": uplift_decimal,
            "uplift_formula": (
                ws_formula.cell(source_row, 7).value
                if isinstance(ws_formula.cell(source_row, 7).value, str)
                and ws_formula.cell(source_row, 7).value.startswith("=")
                else None
            ),
            "source_sheet": ws_values.title,
            "source_row": source_row,
        })

assert recognized_sheets == 8, f"Ожидалось 8 roadmap-листов, найдено {recognized_sheets}"
assert len(roadmap_items) == 204, f"Нарушено авторитетное зерно: {len(roadmap_items)} вместо 204"
assert source_numeric_uplifts == 181, (
    f"Нарушено число item с numeric uplift: {source_numeric_uplifts} вместо 181"
)

print({
    "source_items": len(roadmap_items),
    "items_with_numeric_uplift": source_numeric_uplifts,
    "items_with_formula_uplift": source_formula_uplifts,
    "recognized_sheets": recognized_sheets,
    "sheets_with_items": sorted({item["source_sheet"] for item in roadmap_items}),
    "grain_check": "PASS (204 / 181)",
})

{'source_items': 204, 'items_with_numeric_uplift': 181, 'items_with_formula_uplift': 1, 'recognized_sheets': 8, 'sheets_with_items': ['CX', 'DB', 'DP', 'PC', 'ДомКлик', 'СВР', 'УБ'], 'grain_check': 'PASS (204 / 181)'}


### 5. Map source items and calculate product/block caps

Включённая метрика удовлетворяет всем условиям: `max_value > 0`, `is_applicabble_flg is not false`, `excluded_from_index is not true`, `dd_calculation_flg != 0`.

Формула: `100 × Σ(max_value − value) по включённым метрикам выбранного блока(ов) / Σ(max_value) по всем включённым метрикам продукта`.

In [5]:
def product_scoring(product):
    denominator = Decimal("0")
    by_block = defaultdict(lambda: {"current": Decimal("0"), "maximum": Decimal("0"), "metric_count": 0})
    for block in product.get("metrics", []):
        block_name = normalize_text(block.get("name"))
        for metric in block.get("metrics", []):
            max_value = as_decimal(metric.get("max_value"), allow_none=True)
            included = (
                max_value is not None
                and max_value > 0
                and metric.get("is_applicabble_flg") is not False
                and metric.get("excluded_from_index") is not True
                and metric.get("dd_calculation_flg") != 0
            )
            if not included:
                continue
            value = as_decimal(metric.get("value"))
            denominator += max_value
            by_block[block_name]["current"] += value
            by_block[block_name]["maximum"] += max_value
            by_block[block_name]["metric_count"] += 1
    return denominator, dict(by_block)


product_scores = {key: product_scoring(product) for key, product in product_index.items()}
zero_denominator_products = [key for key, (denominator, _) in product_scores.items() if denominator <= 0]
assert not zero_denominator_products, f"Нулевые знаменатели: {zero_denominator_products}"

mapped_groups = defaultdict(list)
unmapped_items = []
background_unmapped_items = []
for item in roadmap_items:
    block_set, block_reason = canonical_block_set(item["block_source"])
    product_key = (item["unit_json"], item["product_json_name"])
    product = product_index.get(product_key)

    reason = block_reason
    if not item["product_source"]:
        reason = "Пустой продукт"
    elif product is None:
        reason = "Продукт не найден в JSON"
    elif reason is None:
        denominator, block_stats = product_scores[product_key]
        absent_blocks = [block for block in block_set if block not in block_stats]
        if absent_blocks:
            reason = "Блок отсутствует у продукта в JSON: " + ", ".join(absent_blocks)
        elif denominator <= 0:
            reason = "Знаменатель продукта равен нулю"

    if reason is not None:
        unmapped = dict(item)
        unmapped["unmapped_reason"] = reason
        if item["uplift"] is not None:
            unmapped_items.append(unmapped)
        else:
            background_unmapped_items.append(unmapped)
        continue

    group_key = (
        item["unit_source"], item["product_source"], item["unit_json"],
        item["product_json_name"], block_set,
    )
    mapped_groups[group_key].append(item)


group_results = []
for group_key, items in mapped_groups.items():
    unit_source, product_source, unit_json, product_json_name, block_set = group_key
    product_key = (unit_json, product_json_name)
    denominator, block_stats = product_scores[product_key]
    block_current = sum((block_stats[name]["current"] for name in block_set), Decimal("0"))
    block_maximum = sum((block_stats[name]["maximum"] for name in block_set), Decimal("0"))
    gap = block_maximum - block_current
    true_max_raw = HUNDRED * gap / denominator
    excel_total_raw = sum((item["uplift"] or Decimal("0") for item in items), Decimal("0"))
    excel_total_rounded = round_1(excel_total_raw)
    true_max_rounded = round_1(true_max_raw)
    overstatement = excel_total_rounded - true_max_rounded
    source_labels = []
    for item in items:
        if item["block_source"] not in source_labels:
            source_labels.append(item["block_source"])
    group_results.append({
        "unit_source": unit_source,
        "product_source": product_source,
        "unit_json": unit_json,
        "product_json_name": product_json_name,
        "block_set": block_set,
        "block_source_labels": source_labels,
        "items": items,
        "excel_total_raw": excel_total_raw,
        "excel_total_rounded": excel_total_rounded,
        "true_max_raw": true_max_raw,
        "true_max_rounded": true_max_rounded,
        "overstatement": overstatement,
        "block_current": block_current,
        "block_maximum": block_maximum,
        "denominator": denominator,
        "flagged": excel_total_rounded > true_max_rounded,
    })

group_results.sort(key=lambda g: (g["unit_source"], g["product_source"], g["block_set"]))

required_alias_checks = {
    "CX Score": ("Клиентский опыт",),
    "Цели уровня ЛЮ/ЛТ": ("Цели",),
    "Гипотезы": ("Гипотезы и инициативы",),
    "Знание ключевых метрик\nЦели": ("Знание ключевых метрик", "Цели"),
}
for source_label, expected_set in required_alias_checks.items():
    actual_set, reason = canonical_block_set(source_label)
    assert reason is None and actual_set == expected_set, (source_label, actual_set, reason)
assert all(
    any(normalize_text(item["block_source"]) == normalize_text(source_label) for item in roadmap_items)
    for source_label in required_alias_checks
), "Не все обязательные псевдонимы представлены в источнике"

flagged_groups = [group for group in group_results if group["flagged"]]
flagged_detail_items = sum(len(group["items"]) for group in flagged_groups)
unmapped_reason_counts = Counter(item["unmapped_reason"] for item in unmapped_items)
background_unmapped_reason_counts = Counter(
    item["unmapped_reason"] for item in background_unmapped_items
)

numeric_blank_items = [
    item for item in unmapped_items if item["unmapped_reason"] == "Пустой блок"
]
expected_numeric_blank_products = {
    "Детская карта", "Трансграничные переводы", "Трансграничный эквайринг"
}
assert len(numeric_blank_items) == 3
assert {item["product_source"] for item in numeric_blank_items} == expected_numeric_blank_products
assert all(item["uplift"] is not None for item in unmapped_items)
assert len(background_unmapped_items) == 15

print({
    "mapped_groups": len(group_results),
    "flagged_groups": len(flagged_groups),
    "flagged_detail_items": flagged_detail_items,
    "unmapped_numeric_uplift_items": len(unmapped_items),
    "unmapped_numeric_reasons": dict(unmapped_reason_counts),
    "background_unmapped_without_uplift": len(background_unmapped_items),
    "background_unmapped_reasons": dict(background_unmapped_reason_counts),
    "numeric_blank_block_check": sorted(item["product_source"] for item in numeric_blank_items),
})

{'mapped_groups': 147, 'flagged_groups': 19, 'flagged_detail_items': 27, 'unmapped_numeric_uplift_items': 3, 'unmapped_numeric_reasons': {'Пустой блок': 3}, 'background_unmapped_without_uplift': 15, 'background_unmapped_reasons': {'Блок не сопоставлен': 1, 'Пустой блок': 14}, 'numeric_blank_block_check': ['Детская карта', 'Трансграничные переводы', 'Трансграничный эквайринг']}


## Results

### 6. Summary and required spot-check

In [6]:
flagged_excel_sum = sum((group["excel_total_rounded"] for group in flagged_groups), Decimal("0"))
flagged_true_max_sum = sum((group["true_max_rounded"] for group in flagged_groups), Decimal("0"))
flagged_overstatement_sum = sum((group["overstatement"] for group in flagged_groups), Decimal("0"))

summary = {
    "source_items": len(roadmap_items),
    "mapped_groups": len(group_results),
    "flagged_groups": len(flagged_groups),
    "flagged_detail_items": flagged_detail_items,
    "unmapped_numeric_uplift_items": len(unmapped_items),
    "background_unmapped_without_uplift": len(background_unmapped_items),
    "flagged_excel_sum_pp": float(flagged_excel_sum),
    "flagged_true_max_sum_pp": float(flagged_true_max_sum),
    "flagged_overstatement_sum_pp": float(flagged_overstatement_sum),
}
print(summary)

spot_candidates = [
    group for group in group_results
    if group["unit_source"] == "УБ"
    and group["product_source"] == "ДСЖ ПК"
    and group["block_set"] == ("Знание ключевых метрик",)
]
assert len(spot_candidates) == 1, f"Ожидалась одна spot-check группа, найдено {len(spot_candidates)}"
spot_check = spot_candidates[0]
assert spot_check["excel_total_rounded"] == Decimal("30.0")
assert spot_check["true_max_rounded"] == Decimal("5.1")
assert spot_check["flagged"]
print({
    "spot_check": "ДСЖ ПК / Знание ключевых метрик",
    "excel_pp": float(spot_check["excel_total_rounded"]),
    "actual_max_pp": float(spot_check["true_max_rounded"]),
    "block_current_points": float(spot_check["block_current"]),
    "block_max_points": float(spot_check["block_maximum"]),
    "product_denominator": float(spot_check["denominator"]),
    "status": "PASS",
})

print("Топ-10 групп по завышению:")
for group in sorted(flagged_groups, key=lambda g: g["overstatement"], reverse=True)[:10]:
    print(
        group["unit_source"], "|", group["product_source"], "|",
        " + ".join(group["block_set"]), "| Excel", group["excel_total_rounded"],
        "| максимум", group["true_max_rounded"], "| завышение", group["overstatement"]
    )

{'source_items': 204, 'mapped_groups': 147, 'flagged_groups': 19, 'flagged_detail_items': 27, 'unmapped_numeric_uplift_items': 3, 'background_unmapped_without_uplift': 15, 'flagged_excel_sum_pp': 174.9, 'flagged_true_max_sum_pp': 109.1, 'flagged_overstatement_sum_pp': 65.8}
{'spot_check': 'ДСЖ ПК / Знание ключевых метрик', 'excel_pp': 30.0, 'actual_max_pp': 5.1, 'block_current_points': 0.45, 'block_max_points': 1.5, 'product_denominator': 20.75, 'status': 'PASS'}
Топ-10 групп по завышению:
УБ | ИнвестКопилка | Знание ключевых метрик | Excel 30.0 | максимум 3.1 | завышение 26.9
УБ | ДСЖ ПК | Знание ключевых метрик | Excel 30.0 | максимум 5.1 | завышение 24.9
УБ | Страхование залога | Воронка привлечения | Excel 17.7 | максимум 12.4 | завышение 5.3
УБ | ВЗР | Гипотезы и инициативы | Excel 14.8 | максимум 11.9 | завышение 2.9
УБ | БПИФ | Знание ключевых метрик | Excel 5.4 | максимум 3.4 | завышение 2.0
DP | СберKids | Механики | Excel 8.6 | максимум 7.1 | завышение 1.5
УБ | Брокерский сче

### 7. Build the stakeholder workbook

In [7]:
DETAIL_HEADERS = [
    "Продукт",
    "Блок",
    "Метрика / планируемое мероприятие",
    "Аплифт из Excel, п.п.",
    "Настоящий максимум аплифта блока, п.п.",
    "Юнит",
    "Канонический набор блоков",
    "Excel total по блоку, п.п.",
    "Завышение, п.п.",
    "Лист-источник",
    "Строка-источник",
    "Текущие баллы блока",
    "Максимальные баллы блока",
    "Знаменатель продукта",
    "Продукт в JSON",
]

SUMMARY_HEADERS = [
    "Юнит", "Продукт", "Блок", "Канонический набор блоков",
    "Excel total по блоку, п.п.", "Настоящий максимум аплифта блока, п.п.",
    "Завышение, п.п.", "Количество исходных мероприятий",
    "Текущие баллы блока", "Максимальные баллы блока", "Знаменатель продукта",
    "Продукт в JSON", "Листы / строки источника",
]

UNMAPPED_HEADERS = [
    "Юнит", "Продукт", "Блок", "Метрика / планируемое мероприятие",
    "Аплифт из Excel, п.п.", "Причина", "Лист-источник", "Строка-источник",
]

METHOD_ROWS = [
    ("Параметр", "Значение"),
    ("Дата среза / аудита", AUDIT_AS_OF.isoformat()),
    ("Источник дорожных карт", str(SOURCE_XLSX_REL)),
    ("Источник DD-метрик", str(SOURCE_JSON_REL)),
    ("Выходной файл", str(OUTPUT_XLSX_REL)),
    ("Единица дорожной карты", "Логический item по merged-cell semantics build_calc_report.read_roadmap_workbook(); 204 items, из них 181 с numeric uplift."),
    ("Распознавание области", "Листы распознаются по заголовкам B:G; строки заголовков/сводных секций отсеиваются той же логикой, что и в build_calc_report.read_roadmap_workbook()."),
    ("Объединённые ячейки", "Физические строки внутри merge G (или D/F/H для item без uplift) объединяются в одно planned_activity; продукт/блок берутся из top-left merge. Обычные пустые блоки не заполняются."),
    ("Авторитетный grain DD", "Продукт × блок из массива products[].metrics[] файла report-data.json."),
    ("Включённая метрика", "max_value > 0; is_applicabble_flg не равно false; excluded_from_index не равно true; dd_calculation_flg не равно 0."),
    ("Знаменатель продукта", "Сумма max_value всех включённых метрик продукта по всем блокам."),
    ("Теоретический максимум блока", "100 × Σ(max_value − value) включённых метрик сопоставленного блока(ов) / знаменатель продукта."),
    ("Группировка Excel", "Юнит + продукт + канонический набор сопоставленных блоков; Excel total — сумма только числовых expected_uplift в группе."),
    ("Округление", "Decimal ROUND_HALF_UP до 1 десятичного знака отдельно для Excel total и теоретического максимума."),
    ("Флаг", "Округлённый Excel total строго больше округлённого теоретического максимума блока."),
    ("Псевдонимы блоков", "CX Score → Клиентский опыт; Цели уровня ЛЮ/ЛТ → Цели; Гипотезы → Гипотезы и инициативы; многострочный «Знание ключевых метрик + Цели» → объединённый cap двух блоков."),
    ("Псевдонимы юнитов", "; ".join(f"{source} → {target}" for source, target in UNIT_ALIASES.items())),
    ("Псевдонимы продуктов", "; ".join(f"{unit}: {source} → {target}" for (unit, source), target in PRODUCT_ALIASES_RAW.items())),
    ("Несопоставленные строки", "На лист «Не сопоставлено» попадают только logical items с числовым expected_uplift, для которых блок/продукт не сопоставлен; значения не угадываются."),
    ("Фоновая проверка без uplift", f"{len(background_unmapped_items)} несопоставленных logical items без expected_uplift не являются ошибками расчёта и не включены в лист «Не сопоставлено». Причины: {dict(background_unmapped_reason_counts)}."),
    ("Проверка пустого блока с uplift", "3 строки: Детская карта, Трансграничные переводы, Трансграничный эквайринг."),
    ("Ключевая оговорка", "Рассчитанное значение — теоретический максимум для всего блока, а не прогноз конкретного мероприятия."),
    ("Агрегированные суммы", "Суммы по флагам — арифметические суммы строковых групп; при пересекающихся наборах блоков они не являются уникальным портфельным потенциалом."),
]


def decimal_float(value):
    return None if value is None else float(value)


def joined_source_labels(group):
    return " | ".join(group["block_source_labels"])


workbook = Workbook()
workbook.remove(workbook.active)

ws_detail = workbook.create_sheet("Ошибки — детализация")
ws_detail.append(DETAIL_HEADERS)
for group in flagged_groups:
    canonical_label = " + ".join(group["block_set"])
    for item in group["items"]:
        ws_detail.append([
            item["product_source"],
            item["block_source"],
            item["activity"],
            decimal_float(item["uplift"]),
            decimal_float(group["true_max_rounded"]),
            item["unit_source"],
            canonical_label,
            decimal_float(group["excel_total_rounded"]),
            decimal_float(group["overstatement"]),
            item["source_sheet"],
            item["source_row"],
            decimal_float(group["block_current"]),
            decimal_float(group["block_maximum"]),
            decimal_float(group["denominator"]),
            item["product_json_name"],
        ])

ws_summary = workbook.create_sheet("Ошибки — по блокам")
ws_summary.append(SUMMARY_HEADERS)
for group in flagged_groups:
    source_refs = "; ".join(
        f'{item["source_sheet"]}!{item["source_row"]}' for item in group["items"]
    )
    ws_summary.append([
        group["unit_source"],
        group["product_source"],
        joined_source_labels(group),
        " + ".join(group["block_set"]),
        decimal_float(group["excel_total_rounded"]),
        decimal_float(group["true_max_rounded"]),
        decimal_float(group["overstatement"]),
        len(group["items"]),
        decimal_float(group["block_current"]),
        decimal_float(group["block_maximum"]),
        decimal_float(group["denominator"]),
        group["product_json_name"],
        source_refs,
    ])

ws_unmapped = workbook.create_sheet("Не сопоставлено")
ws_unmapped.append(UNMAPPED_HEADERS)
for item in sorted(unmapped_items, key=lambda x: (x["source_sheet"], x["source_row"])):
    ws_unmapped.append([
        item["unit_source"], item["product_source"], item["block_source"], item["activity"],
        decimal_float(item["uplift"]), item["unmapped_reason"], item["source_sheet"], item["source_row"],
    ])

ws_method = workbook.create_sheet("Методика")
for row in METHOD_ROWS:
    ws_method.append(row)

header_fill = PatternFill("solid", fgColor="0B6B50")
header_font = Font(color="FFFFFF", bold=True)
red_fill = PatternFill("solid", fgColor="FFC7CE")
thin_gray = Side(style="thin", color="D9E2E3")
border = Border(left=thin_gray, right=thin_gray, top=thin_gray, bottom=thin_gray)

for ws in workbook.worksheets:
    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions
    ws.sheet_view.showGridLines = False
    ws.row_dimensions[1].height = 32
    for cell in ws[1]:
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        cell.border = border
    for row in ws.iter_rows(min_row=2):
        for cell in row:
            cell.alignment = Alignment(vertical="top", wrap_text=True)
            cell.border = border

# Numeric formats.
for ws, one_decimal_headers, three_decimal_headers in [
    (ws_detail,
     {"Аплифт из Excel, п.п.", "Настоящий максимум аплифта блока, п.п.", "Excel total по блоку, п.п.", "Завышение, п.п."},
     {"Текущие баллы блока", "Максимальные баллы блока", "Знаменатель продукта"}),
    (ws_summary,
     {"Excel total по блоку, п.п.", "Настоящий максимум аплифта блока, п.п.", "Завышение, п.п."},
     {"Текущие баллы блока", "Максимальные баллы блока", "Знаменатель продукта"}),
    (ws_unmapped, {"Аплифт из Excel, п.п."}, set()),
]:
    header_to_col = {cell.value: cell.column for cell in ws[1]}
    for header in one_decimal_headers:
        col = header_to_col[header]
        for row in range(2, ws.max_row + 1):
            ws.cell(row, col).number_format = "0.0"
    for header in three_decimal_headers:
        col = header_to_col[header]
        for row in range(2, ws.max_row + 1):
            ws.cell(row, col).number_format = "0.000"

# Conditional red highlighting for overstatement.
for ws in (ws_detail, ws_summary):
    header_to_col = {cell.value: cell.column for cell in ws[1]}
    overstatement_col = header_to_col["Завышение, п.п."]
    overstatement_letter = get_column_letter(overstatement_col)
    if ws.max_row >= 2:
        ws.conditional_formatting.add(
            f"{overstatement_letter}2:{overstatement_letter}{ws.max_row}",
            CellIsRule(operator="greaterThan", formula=["0"], fill=red_fill),
        )

# Readable widths with bounded auto-sizing and explicit text-column priorities.
for ws in workbook.worksheets:
    for column_cells in ws.columns:
        letter = get_column_letter(column_cells[0].column)
        max_len = max((len(str(cell.value)) for cell in column_cells if cell.value is not None), default=0)
        ws.column_dimensions[letter].width = min(max(max_len + 2, 10), 36)

for ws in (ws_detail, ws_unmapped):
    ws.column_dimensions["C"].width = 30 if ws is ws_detail else 26
    activity_col = 3 if ws is ws_detail else 4
    ws.column_dimensions[get_column_letter(activity_col)].width = 72
ws_summary.column_dimensions["C"].width = 28
ws_summary.column_dimensions["D"].width = 34
ws_summary.column_dimensions["M"].width = 38
ws_method.column_dimensions["A"].width = 34
ws_method.column_dimensions["B"].width = 110

for ws in (ws_detail, ws_summary, ws_unmapped):
    for row_number in range(2, ws.max_row + 1):
        ws.row_dimensions[row_number].height = 46
for row_number in range(2, ws_method.max_row + 1):
    ws_method.row_dimensions[row_number].height = 44

workbook.save(OUTPUT_XLSX)
print(f"Saved {OUTPUT_XLSX_REL} ({OUTPUT_XLSX.stat().st_size:,} bytes)")

Saved artifacts/Аудит_аплифта_дорожных_карт_DD.xlsx (17,130 bytes)


### 8. Reopen and validate workbook formulas/values

In [8]:
formula_view = load_workbook(OUTPUT_XLSX, data_only=False, read_only=False)
value_view = load_workbook(OUTPUT_XLSX, data_only=True, read_only=False)
expected_sheets = ["Ошибки — детализация", "Ошибки — по блокам", "Не сопоставлено", "Методика"]
assert formula_view.sheetnames == expected_sheets
assert value_view.sheetnames == expected_sheets

assert [cell.value for cell in formula_view[expected_sheets[0]][1]][:6] == DETAIL_HEADERS[:6]
assert formula_view[expected_sheets[0]].max_row - 1 == flagged_detail_items
assert formula_view[expected_sheets[1]].max_row - 1 == len(flagged_groups)
assert formula_view[expected_sheets[2]].max_row - 1 == len(unmapped_items)

formula_cells = []
for ws in formula_view.worksheets:
    for row in ws.iter_rows():
        for cell in row:
            if isinstance(cell.value, str) and cell.value.startswith("="):
                formula_cells.append((ws.title, cell.coordinate, cell.value))
assert not formula_cells, "Выходной файл должен содержать зафиксированные значения, а не формулы без кэша"

# Reopen spot-check in both formula and value modes.
for reopened in (formula_view, value_view):
    ws = reopened["Ошибки — по блокам"]
    headers = {cell.value: cell.column for cell in ws[1]}
    rows = [
        row for row in range(2, ws.max_row + 1)
        if ws.cell(row, headers["Продукт"]).value == "ДСЖ ПК"
        and ws.cell(row, headers["Канонический набор блоков"]).value == "Знание ключевых метрик"
    ]
    assert len(rows) == 1
    row = rows[0]
    assert ws.cell(row, headers["Excel total по блоку, п.п."]).value == 30
    assert ws.cell(row, headers["Настоящий максимум аплифта блока, п.п."]).value == 5.1

validation = {
    "status": "PASS",
    "sheets": formula_view.sheetnames,
    "detail_rows": formula_view[expected_sheets[0]].max_row - 1,
    "block_rows": formula_view[expected_sheets[1]].max_row - 1,
    "unmapped_rows": formula_view[expected_sheets[2]].max_row - 1,
    "output_formula_cells": len(formula_cells),
    "reopened_data_only_false": True,
    "reopened_data_only_true": True,
    "spot_check": "30.0 vs 5.1",
}
print(validation)

{'status': 'PASS', 'sheets': ['Ошибки — детализация', 'Ошибки — по блокам', 'Не сопоставлено', 'Методика'], 'detail_rows': 27, 'block_rows': 19, 'unmapped_rows': 3, 'output_formula_cells': 0, 'reopened_data_only_false': True, 'reopened_data_only_true': True, 'spot_check': '30.0 vs 5.1'}


## Takeaways

- Флаг означает только математическое превышение заявленной суммы над оставшимся теоретическим максимумом блока после требуемого округления.
- Строки с пустым или неизвестным блоком не включены в сравнение и вынесены на лист «Не сопоставлено».
- Рассчитанный максимум относится ко всему блоку; его нельзя интерпретировать как прогноз эффекта отдельного мероприятия.